In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path("../data/raw")
REPORT_DIR = Path("../reports")

REPORT_DIR.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(DATA_DIR / "train.csv")
campaigns = pd.read_csv(DATA_DIR / "campaign_data.csv")
customers = pd.read_csv(DATA_DIR / "customer_demographics.csv")

print("Datasets loaded successfully!")
print("-" * 45)

print("Training data:", train.shape)
print("Campaign data:", campaigns.shape)
print("Customer demographics:", customers.shape)

Datasets loaded successfully!
---------------------------------------------
Training data: (78369, 5)
Campaign data: (28, 4)
Customer demographics: (760, 7)


In [2]:
# Count campaigns associated with each customer

customer_campaign_counts = (
    train.groupby("customer_id")
    .agg(
        campaign_count=("campaign_id", "nunique"),
        coupon_count=("coupon_id", "nunique"),
        total_records=("id", "count")
    )
    .reset_index()
)

print("Customer campaign summary:")
display(customer_campaign_counts.head(10))


# Summarize the number of campaigns per customer

print("\nDistribution of campaigns per customer:")

campaign_distribution = (
    customer_campaign_counts["campaign_count"]
    .value_counts()
    .sort_index()
    .reset_index()
)

campaign_distribution.columns = [
    "campaigns_per_customer",
    "number_of_customers"
]

display(campaign_distribution)


# Overall statistics

print("\nOverall Summary")
print("-" * 40)

print(
    "Unique customers:",
    customer_campaign_counts["customer_id"].nunique()
)

print(
    "Customers appearing in multiple campaigns:",
    (customer_campaign_counts["campaign_count"] > 1).sum()
)

print(
    "Customers appearing in only one campaign:",
    (customer_campaign_counts["campaign_count"] == 1).sum()
)

Customer campaign summary:


,customer_id,campaign_count,coupon_count,total_records
0,1,4,93,94
1,3,2,42,42
2,4,1,23,23
3,5,3,52,52
4,6,2,32,32
5,7,7,118,125
6,8,6,93,98
7,9,2,43,43
8,10,2,36,36
9,11,4,70,72



Distribution of campaigns per customer:


,campaigns_per_customer,number_of_customers
0,1,312
1,2,343
2,3,284
3,4,217
4,5,157
5,6,73
6,7,25
7,8,13
8,9,2
9,10,1



Overall Summary
----------------------------------------
Unique customers: 1428
Customers appearing in multiple campaigns: 1116
Customers appearing in only one campaign: 312


In [3]:
# 1. Count unique coupons in each campaign

campaign_coupon_summary = (
    train.groupby("campaign_id")
    .agg(
        total_records=("id", "count"),
        unique_coupons=("coupon_id", "nunique"),
        unique_customers=("customer_id", "nunique")
    )
    .reset_index()
    .sort_values("unique_coupons", ascending=False)
)

print("Coupon distribution by campaign:")
display(campaign_coupon_summary)


# 2. Check how many campaigns each coupon appears in

coupon_campaign_counts = (
    train.groupby("coupon_id")["campaign_id"]
    .nunique()
    .reset_index(name="campaign_count")
)

print("\nDistribution of campaigns per coupon:")
display(
    coupon_campaign_counts["campaign_count"]
    .value_counts()
    .sort_index()
    .rename_axis("campaigns_per_coupon")
    .reset_index(name="number_of_coupons")
)


# 3. Overall summary

print("\nOverall Summary")
print("-" * 40)

print("Unique coupons:", train["coupon_id"].nunique())

print(
    "Coupons appearing in multiple campaigns:",
    (coupon_campaign_counts["campaign_count"] > 1).sum()
)

print(
    "Coupons appearing in only one campaign:",
    (coupon_campaign_counts["campaign_count"] == 1).sum()
)

Coupon distribution by campaign:


,campaign_id,total_records,unique_coupons,unique_customers
7,8,22555,208,1076
12,13,22606,207,1077
13,26,6056,181,332
17,30,6467,178,361
2,3,408,34,12
16,29,3895,33,118
15,28,476,28,17
14,27,324,27,12
8,9,3168,18,176
1,2,768,16,48



Distribution of campaigns per coupon:


,campaigns_per_coupon,number_of_coupons
0,1,752
1,2,85
2,3,24
3,4,4
4,5,1



Overall Summary
----------------------------------------
Unique coupons: 866
Coupons appearing in multiple campaigns: 114
Coupons appearing in only one campaign: 752


In [4]:
# 1. Check duplicate customer-campaign combinations

customer_campaign_duplicates = train.duplicated(
    subset=["customer_id", "campaign_id"]
).sum()

print("Customer-campaign relationship check")
print("-" * 45)

print(
    "Duplicate customer-campaign combinations:",
    customer_campaign_duplicates
)


# 2. Check duplicate customer-campaign-coupon combinations

customer_campaign_coupon_duplicates = train.duplicated(
    subset=["customer_id", "campaign_id", "coupon_id"]
).sum()

print(
    "\nDuplicate customer-campaign-coupon combinations:",
    customer_campaign_coupon_duplicates
)


# 3. Check whether the training ID is unique

print("\nTraining ID validation")
print("-" * 45)

print("Total training records:", len(train))
print("Unique training IDs:", train["id"].nunique())
print("Missing training IDs:", train["id"].isna().sum())


# 4. Summarize the validation results

relationship_validation = pd.DataFrame([
    {
        "Check": "Customer-campaign combinations",
        "Duplicate_count": int(customer_campaign_duplicates)
    },
    {
        "Check": "Customer-campaign-coupon combinations",
        "Duplicate_count": int(customer_campaign_coupon_duplicates)
    },
    {
        "Check": "Training IDs",
        "Duplicate_count": int(train["id"].duplicated().sum())
    }
])

display(relationship_validation)

Customer-campaign relationship check
---------------------------------------------
Duplicate customer-campaign combinations: 74110

Duplicate customer-campaign-coupon combinations: 12

Training ID validation
---------------------------------------------
Total training records: 78369
Unique training IDs: 78369
Missing training IDs: 0


,Check,Duplicate_count
0,Customer-campaign combinations,74110
1,Customer-campaign-coupon combinations,12
2,Training IDs,0


In [5]:
# Save customer-campaign relationship analysis

customer_campaign_counts.to_csv(
    REPORT_DIR / "day10_customer_campaign_counts.csv",
    index=False
)

campaign_coupon_summary.to_csv(
    REPORT_DIR / "day10_campaign_coupon_summary.csv",
    index=False
)

coupon_campaign_counts.to_csv(
    REPORT_DIR / "day10_coupon_campaign_counts.csv",
    index=False
)

relationship_validation.to_csv(
    REPORT_DIR / "day10_relationship_validation.csv",
    index=False
)

print("Day 10 analysis files saved successfully!")

print("\nFiles created:")
print("1. day10_customer_campaign_counts.csv")
print("2. day10_campaign_coupon_summary.csv")
print("3. day10_coupon_campaign_counts.csv")
print("4. day10_relationship_validation.csv")

Day 10 analysis files saved successfully!

Files created:
1. day10_customer_campaign_counts.csv
2. day10_campaign_coupon_summary.csv
3. day10_coupon_campaign_counts.csv
4. day10_relationship_validation.csv
